# Drift Evaluation — Visualización y Comparación (con y sin 1H)
 
Este notebook:
- NO recalcula drift.
- Lee resultados exportados por el notebook core:
     - synthetic_data/results
     - synthetic_data/results_1H
- Reconstruye episodios manuales desde synthetic_plant_events.csv.
- Genera:
     1) Resúmenes + gráficos para el caso base (ventanas ≥ 6H).
     2) Episodios refinados usando detecciones de 1H.
     3) Nueva evaluación con episodios refinados.
     4) Comparación directa base vs refinado.

 Supone que la lógica de evaluación en el notebook core ya se ejecutó y generó todos los CSV de ambas carpetas.

In [14]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px

# Paths base (ajusta si cambiaste estructura)
BASE_DIR        = Path("synthetic_data")
RESULTS_DIR     = BASE_DIR / "results"
RESULTS_1H_DIR  = BASE_DIR / "results_1H"   # en tu estructura está anidado dentro de results
METRICS_DIR     = RESULTS_DIR / "metrics"
METRICS_1H_DIR  = RESULTS_1H_DIR / "metrics"

SERIES_PATH     = BASE_DIR / "synthetic_plant.csv"
LABELS_PATH     = BASE_DIR / "synthetic_plant_events.csv"

print("RESULTS_DIR:   ", RESULTS_DIR.resolve())
print("RESULTS_1H_DIR:", RESULTS_1H_DIR.resolve())

RESULTS_DIR:    C:\Users\frncc\OneDrive - Universidad Católica de Chile\Desktop\UC\2025-2\Proyecto de Grado\Proyecto-Grado\Drift Evaluation\synthetic_data\results
RESULTS_1H_DIR: C:\Users\frncc\OneDrive - Universidad Católica de Chile\Desktop\UC\2025-2\Proyecto de Grado\Proyecto-Grado\Drift Evaluation\synthetic_data\results_1H


## 2. Carga de serie y episodios manuales

In [15]:
# Carga de serie sintética
df_raw = pd.read_csv(SERIES_PATH)

if "date_time" not in df_raw.columns:
    raise ValueError("synthetic_plant.csv debe tener columna 'date_time'.")

df_raw["date_time"] = pd.to_datetime(df_raw["date_time"], errors="coerce")
df_raw = (
    df_raw
    .dropna(subset=["date_time"])
    .sort_values("date_time")
    .set_index("date_time")
)

df = df_raw.select_dtypes(include="number").copy()
assert not df.empty, "No hay columnas numéricas en la serie sintética."

t_min, t_max = df.index.min(), df.index.max()
print(f"Rango temporal de la serie: {t_min}  →  {t_max}")
print("N° filas:", len(df), " | N° vars numéricas:", df.shape[1])

# Carga de eventos manuales
events = pd.read_csv(LABELS_PATH)
events["date_time"] = pd.to_datetime(events["date_time"], errors="coerce")
events = (
    events
    .dropna(subset=["date_time", "variable", "event"])
    .assign(event=lambda s: s["event"].str.lower().str.strip())
    .query("event in ['start','end']")
    .sort_values(["variable", "date_time"])
    .reset_index(drop=True)
)

def events_to_intervals(ev: pd.DataFrame) -> pd.DataFrame:
    has_type = "drift_type" in ev.columns
    rows = []
    for var, g in ev.groupby("variable", sort=True):
        open_t = None
        open_type = None
        for _, r in g.iterrows():
            evt = str(r["event"]).lower()
            dt_val = r["drift_type"] if has_type else "unknown"

            if evt == "start":
                open_t = r["date_time"]
                open_type = dt_val
            elif evt == "end" and open_t is not None and r["date_time"] > open_t:
                rows.append({
                    "variable": var,
                    "manual_start": open_t,
                    "manual_end": r["date_time"],
                    "drift_type": open_type,
                })
                open_t = None
                open_type = None
    return pd.DataFrame(rows)

intervals_manual = events_to_intervals(events)
print("Total episodios manuales:", len(intervals_manual))
if "drift_type" in intervals_manual.columns:
    print("\nDistribución por drift_type:")
    print(intervals_manual["drift_type"].value_counts())

Rango temporal de la serie: 2025-01-01 00:00:00  →  2025-01-20 23:59:00
N° filas: 28800  | N° vars numéricas: 10
Total episodios manuales: 23

Distribución por drift_type:
drift_type
gradual    15
abrupt      8
Name: count, dtype: int64


In [16]:

# %% 3. Carga de resultados base (ventanas ≥ 6H)

eval_path   = RESULTS_DIR / "eval_episodes_by_window_strategy.csv"
manual_path = RESULTS_DIR / "manual_marked_episodes.csv"
auto_path   = RESULTS_DIR / "auto_marked_episodes.csv"

eval_df = pd.read_csv(eval_path)
manual_marked = pd.read_csv(manual_path)
auto_marked   = pd.read_csv(auto_path)

print("eval_df shape:", eval_df.shape)
print("auto_marked shape:", auto_marked.shape)
display(eval_df.head())

# Normalizamos tipos y recomputamos Score_F1_combined = 0.5 * F1 + 0.5 * F1_time
eval_df = eval_df.copy()
eval_df["window"]   = eval_df["window"].astype(str)
eval_df["strategy"] = eval_df["strategy"].astype(str)
eval_df["metric"]   = eval_df["metric"].astype(str)

eval_df["F1_time_filled"] = eval_df["F1_time"].fillna(0.0)
eval_df["Score_F1_combined"] = 0.5 * eval_df["F1"] + 0.5 * eval_df["F1_time_filled"]

display(eval_df.head())

eval_df shape: (45, 24)
auto_marked shape: (748, 12)


,window,strategy,metric,TP_episodes,FP_episodes,FN_episodes,Precision,Recall,F1,manual_total_hours,...,F1_time,coverage_mean,coverage_median,delay_mean_hours,delay_median_hours,false_alarms_per_day,extra_hours,extra_ratio_auto,F1_time_filled,Score_F1_combined
0,12H,decay,ks,22,1,1,0.956522,0.956522,0.956522,790.366667,...,0.631425,0.923201,1.000000,7.294697,6.000000,0.050002,834.383333,0.526757,0.631425,0.793973
1,12H,decay,psi,22,1,1,0.956522,0.956522,0.956522,790.366667,...,0.631121,0.916111,1.000000,7.655303,6.000000,0.050002,818.316667,0.524562,0.631121,0.793822
2,12H,decay,wasserstein,22,1,1,0.956522,0.956522,0.956522,790.366667,...,0.647655,0.916111,1.000000,7.655303,6.000000,0.050002,758.316667,0.505544,0.647655,0.802088
3,12H,golden,ks,22,2,1,0.916667,0.956522,0.936170,790.366667,...,0.683573,0.871735,1.000000,8.684091,6.000000,0.100003,535.516667,0.437514,0.683573,0.809872
4,12H,golden,psi,22,2,1,0.916667,0.956522,0.936170,790.366667,...,0.642938,0.807490,0.905717,11.362879,6.408333,0.100003,462.450000,0.437926,0.642938,0.789554


,window,strategy,metric,TP_episodes,FP_episodes,FN_episodes,Precision,Recall,F1,manual_total_hours,...,F1_time,coverage_mean,coverage_median,delay_mean_hours,delay_median_hours,false_alarms_per_day,extra_hours,extra_ratio_auto,F1_time_filled,Score_F1_combined
0,12H,decay,ks,22,1,1,0.956522,0.956522,0.956522,790.366667,...,0.631425,0.923201,1.000000,7.294697,6.000000,0.050002,834.383333,0.526757,0.631425,0.793973
1,12H,decay,psi,22,1,1,0.956522,0.956522,0.956522,790.366667,...,0.631121,0.916111,1.000000,7.655303,6.000000,0.050002,818.316667,0.524562,0.631121,0.793822
2,12H,decay,wasserstein,22,1,1,0.956522,0.956522,0.956522,790.366667,...,0.647655,0.916111,1.000000,7.655303,6.000000,0.050002,758.316667,0.505544,0.647655,0.802088
3,12H,golden,ks,22,2,1,0.916667,0.956522,0.936170,790.366667,...,0.683573,0.871735,1.000000,8.684091,6.000000,0.100003,535.516667,0.437514,0.683573,0.809872
4,12H,golden,psi,22,2,1,0.916667,0.956522,0.936170,790.366667,...,0.642938,0.807490,0.905717,11.362879,6.408333,0.100003,462.450000,0.437926,0.642938,0.789554


## 4. Carga de resultados SOLO 1H

In [17]:
eval_1h_path   = RESULTS_1H_DIR / "eval_episodes_by_window_strategy.csv"
manual_1h_path = RESULTS_1H_DIR / "manual_marked_episodes.csv"
auto_1h_path   = RESULTS_1H_DIR / "auto_marked_episodes.csv"

eval_1h_df   = pd.read_csv(eval_1h_path)
manual_1h_df = pd.read_csv(manual_1h_path)
auto_1h_df   = pd.read_csv(auto_1h_path)

print("eval_1h_df shape:", eval_1h_df.shape)
print("auto_1h_df shape:", auto_1h_df.shape)
display(eval_1h_df.head())

eval_1h_df = eval_1h_df.copy()
eval_1h_df["window"]   = eval_1h_df["window"].astype(str)
eval_1h_df["strategy"] = eval_1h_df["strategy"].astype(str)
eval_1h_df["metric"]   = eval_1h_df["metric"].astype(str)

eval_1h_df["F1_time_filled"] = eval_1h_df["F1_time"].fillna(0.0)
eval_1h_df["Score_F1_combined"] = 0.5 * eval_1h_df["F1"] + 0.5 * eval_1h_df["F1_time_filled"]

display(eval_1h_df.head())

eval_1h_df shape: (9, 24)
auto_1h_df shape: (4112, 12)


,window,strategy,metric,TP_episodes,FP_episodes,FN_episodes,Precision,Recall,F1,manual_total_hours,...,F1_time,coverage_mean,coverage_median,delay_mean_hours,delay_median_hours,false_alarms_per_day,extra_hours,extra_ratio_auto,F1_time_filled,Score_F1_combined
0,1H,decay,ks,23,450,0,0.048626,1.0,0.092742,790.366667,...,0.383555,0.961849,1.000000,6.096377,6.0,22.500781,2463.833333,0.761382,0.383555,0.238149
1,1H,decay,psi,23,331,0,0.064972,1.0,0.122016,790.366667,...,0.334817,0.964859,1.000000,6.147101,6.0,16.550575,3069.000000,0.798179,0.334817,0.228417
2,1H,decay,wasserstein,23,435,0,0.050218,1.0,0.095634,790.366667,...,0.487116,0.903026,0.968321,6.626812,6.0,21.750755,1514.033333,0.671114,0.487116,0.291375
3,1H,golden,ks,23,505,0,0.043561,1.0,0.083485,790.366667,...,0.346337,0.882193,0.915966,6.215942,6.0,25.250877,2391.583333,0.782074,0.346337,0.214911
4,1H,golden,psi,23,299,0,0.071429,1.0,0.133333,790.366667,...,0.312945,0.944550,0.982493,6.171014,6.0,14.950519,3190.550000,0.812051,0.312945,0.223139


,window,strategy,metric,TP_episodes,FP_episodes,FN_episodes,Precision,Recall,F1,manual_total_hours,...,F1_time,coverage_mean,coverage_median,delay_mean_hours,delay_median_hours,false_alarms_per_day,extra_hours,extra_ratio_auto,F1_time_filled,Score_F1_combined
0,1H,decay,ks,23,450,0,0.048626,1.0,0.092742,790.366667,...,0.383555,0.961849,1.000000,6.096377,6.0,22.500781,2463.833333,0.761382,0.383555,0.238149
1,1H,decay,psi,23,331,0,0.064972,1.0,0.122016,790.366667,...,0.334817,0.964859,1.000000,6.147101,6.0,16.550575,3069.000000,0.798179,0.334817,0.228417
2,1H,decay,wasserstein,23,435,0,0.050218,1.0,0.095634,790.366667,...,0.487116,0.903026,0.968321,6.626812,6.0,21.750755,1514.033333,0.671114,0.487116,0.291375
3,1H,golden,ks,23,505,0,0.043561,1.0,0.083485,790.366667,...,0.346337,0.882193,0.915966,6.215942,6.0,25.250877,2391.583333,0.782074,0.346337,0.214911
4,1H,golden,psi,23,299,0,0.071429,1.0,0.133333,790.366667,...,0.312945,0.944550,0.982493,6.171014,6.0,14.950519,3190.550000,0.812051,0.312945,0.223139


## 5. Resúmenes agregados (caso base, sin refinar con 1H)

In [18]:
cols_summary = [
    c for c in [
        "F1", "F1_time", "delay_mean_hours",
        "extra_ratio_auto", "false_alarms_per_day",
        "Score_F1_combined"
    ]
    if c in eval_df.columns
]

window_order   = sorted(eval_df["window"].unique(), key=lambda w: int(str(w).rstrip("H")))
method_order   = ["psi", "ks", "wasserstein"]
strategy_order = ["decay", "seasonal", "golden"]

summary_full_base = (
    eval_df
    .groupby(["metric", "window", "strategy"])[cols_summary]
    .mean()
    .reset_index()
)

summary_full_base["metric"]   = pd.Categorical(summary_full_base["metric"],   categories=method_order,   ordered=True)
summary_full_base["window"]   = pd.Categorical(summary_full_base["window"],   categories=window_order,   ordered=True)
summary_full_base["strategy"] = pd.Categorical(summary_full_base["strategy"], categories=strategy_order, ordered=True)

summary_full_base = summary_full_base.sort_values(["metric", "window", "strategy"])

print("📘 Resumen base por método, ventana y estrategia")
display(summary_full_base.head(40))

summary_by_method_base = (
    summary_full_base
    .groupby("metric")[cols_summary]
    .mean()
    .reset_index()
    .sort_values("metric")
)

summary_by_window_base = (
    summary_full_base
    .groupby("window")[cols_summary]
    .mean()
    .reset_index()
    .sort_values("window", key=lambda s: s.astype(str).str.rstrip("H").astype(int))
)

summary_by_strategy_base = (
    summary_full_base
    .groupby("strategy")[cols_summary]
    .mean()
    .reset_index()
    .sort_values("strategy")
)

print("\n📌 Promedios base por ventana:")
display(summary_by_window_base)

print("\n📌 Promedios base por método estadístico:")
display(summary_by_method_base)

print("\n📌 Promedios base por estrategia:")
display(summary_by_strategy_base)


📘 Resumen base por método, ventana y estrategia


,metric,window,strategy,F1,F1_time,delay_mean_hours,extra_ratio_auto,false_alarms_per_day,Score_F1_combined
27,psi,6H,decay,0.893617,0.631283,7.903968,0.525053,0.150005,0.762450
29,psi,6H,seasonal,0.893617,0.627913,7.285714,0.532870,0.150005,0.760765
28,psi,6H,golden,0.750000,0.627031,10.761111,0.469123,0.600021,0.688516
15,psi,12H,decay,0.956522,0.631121,7.655303,0.524562,0.050002,0.793822
17,psi,12H,seasonal,0.956522,0.624742,7.655303,0.531766,0.050002,0.790632
16,psi,12H,golden,0.936170,0.642938,11.362879,0.437926,0.100003,0.789554
18,psi,24H,decay,0.977778,0.580732,9.097727,0.562521,0.000000,0.779255
20,psi,24H,seasonal,0.977778,0.574862,9.097727,0.569150,0.000000,0.776320
19,psi,24H,golden,0.954545,0.643511,8.830159,0.470479,0.000000,0.799028
21,psi,36H,decay,0.977778,0.555165,8.184848,0.602923,0.000000,0.766471



📌 Promedios base por ventana:


C:\Users\frncc\AppData\Local\Temp\ipykernel_15460\230615209.py:32: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\frncc\AppData\Local\Temp\ipykernel_15460\230615209.py:40: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\frncc\AppData\Local\Temp\ipykernel_15460\230615209.py:48: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,window,F1,F1_time,delay_mean_hours,extra_ratio_auto,false_alarms_per_day,Score_F1_combined
0,6H,0.833135,0.634487,8.158233,0.503423,0.394458,0.733811
1,12H,0.949738,0.644308,8.276347,0.494344,0.066669,0.797023
2,24H,0.970258,0.618126,7.894140,0.523321,0.005556,0.794192
3,36H,0.990011,0.582246,7.863500,0.573777,0.000000,0.786129
4,48H,0.846561,0.534839,7.012854,0.589242,0.061113,0.690700



📌 Promedios base por método estadístico:


,metric,F1,F1_time,delay_mean_hours,extra_ratio_auto,false_alarms_per_day,Score_F1_combined
0,psi,0.913671,0.594077,8.513797,0.541225,0.090003,0.753874
1,ks,0.902180,0.605031,7.330198,0.539541,0.166672,0.753606
2,wasserstein,0.937971,0.609295,7.679049,0.529698,0.060002,0.773633



📌 Promedios base por estrategia:


,strategy,F1,F1_time,delay_mean_hours,extra_ratio_auto,false_alarms_per_day,Score_F1_combined
0,decay,0.928164,0.594418,7.542972,0.555188,0.070002,0.761291
1,seasonal,0.933510,0.586636,7.410331,0.564468,0.063336,0.760073
2,golden,0.892148,0.627350,8.569741,0.490809,0.183340,0.759749


## 6. Visualizaciones compactas (caso base)

In [19]:
eval_plot_base = eval_df.copy()
eval_plot_base = eval_plot_base[eval_plot_base["metric"].isin(method_order)].copy()
eval_plot_base["method_label"] = eval_plot_base["metric"].map({
    "psi": "PSI",
    "ks": "Kolmogorov–Smirnov",
    "wasserstein": "Wasserstein"
})

eval_plot_base["window"] = pd.Categorical(eval_plot_base["window"], categories=window_order, ordered=True)
eval_plot_base["strategy"] = pd.Categorical(eval_plot_base["strategy"], categories=strategy_order, ordered=True)
eval_plot_base["method_label"] = pd.Categorical(
    eval_plot_base["method_label"],
    categories=["PSI", "Kolmogorov–Smirnov", "Wasserstein"],
    ordered=True
)

fig_score_base = px.bar(
    eval_plot_base,
    x="window",
    y="Score_F1_combined",
    color="strategy",
    barmode="group",
    facet_col="method_label",
    category_orders={
        "window": window_order,
        "strategy": strategy_order,
        "method_label": ["PSI", "Kolmogorov–Smirnov", "Wasserstein"],
    },
    labels={
        "window": "Ventana",
        "Score_F1_combined": "Score combinado (F1 / F1_time)",
        "strategy": "Estrategia",
        "method_label": "Método estadístico",
    },
    title="Score combinado (BASE) por ventana, estrategia y método",
)
fig_score_base.update_yaxes(matches=None)
fig_score_base.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_score_base.show()

fig_delay_base = px.bar(
    eval_plot_base,
    x="window",
    y="delay_mean_hours",
    color="strategy",
    barmode="group",
    facet_col="method_label",
    labels={"delay_mean_hours": "Delay medio (horas)"},
    category_orders={
        "window": window_order,
        "strategy": strategy_order,
        "method_label": ["PSI", "Kolmogorov–Smirnov", "Wasserstein"],
    },
    title="Delay medio (BASE) por ventana, estrategia y método",
)
fig_delay_base.update_yaxes(matches=None)
fig_delay_base.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_delay_base.show()

fig_extra_base = px.bar(
    eval_plot_base,
    x="window",
    y="extra_ratio_auto",
    color="strategy",
    barmode="group",
    facet_col="method_label",
    labels={"extra_ratio_auto": "Proporción de tiempo extra"},
    category_orders={
        "window": window_order,
        "strategy": strategy_order,
        "method_label": ["PSI", "Kolmogorov–Smirnov", "Wasserstein"],
    },
    title="Tiempo extra (BASE) por ventana, estrategia y método",
)
fig_extra_base.update_yaxes(matches=None)
fig_extra_base.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_extra_base.show()


## 7. Construir episodios refinados usando 1H

In [20]:
# Parseo de tiempos y creación de seg_length
for df_eps in [auto_marked, auto_1h_df]:
    for col in ["seg_start", "seg_end"]:
        if col in df_eps.columns:
            df_eps[col] = pd.to_datetime(df_eps[col], errors="coerce")
    if "seg_length" not in df_eps.columns and {"seg_start", "seg_end"} <= set(df_eps.columns):
        df_eps["seg_length"] = df_eps["seg_end"] - df_eps["seg_start"]

print("auto_marked cols:", auto_marked.columns.tolist())
print("auto_1h_df cols:", auto_1h_df.columns.tolist())

def refine_episodes_with_1h(episodes_all: pd.DataFrame,
                            episodes_1h: pd.DataFrame) -> pd.DataFrame:
    """Ajusta episodios de ventanas grandes usando episodios con ventana 1H."""
    eps_1h = episodes_1h.copy()
    eps_big = episodes_all.copy()

    # nos aseguramos que eps_big no incluya 1H
    if "window" in eps_big.columns:
        eps_big = eps_big[eps_big["window"] != "1H"].copy()

    if eps_1h.empty:
        print("⚠️ No hay episodios 1H; no se puede refinar.")
        return episodes_all

    refined_rows = []

    for _, row in eps_big.iterrows():
        var    = row["variable"]
        strat  = row.get("strategy", None)
        metric = row.get("metric", None)
        start_b = row["seg_start"]
        end_b   = row["seg_end"]

        cand = eps_1h[
            (eps_1h["variable"] == var) &
            (eps_1h["strategy"] == strat) &
            (eps_1h["metric"]   == metric)
        ]

        if cand.empty:
            row_ref = row.copy()
            row_ref["seg_start_refined"]  = row["seg_start"]
            row_ref["seg_length_refined"] = row["seg_length"]
            refined_rows.append(row_ref)
            continue

        mask_overlap = ~(
            (cand["seg_end"]   < start_b) |
            (cand["seg_start"] > end_b)
        )
        overlap_1h = cand[mask_overlap]

        if overlap_1h.empty:
            row_ref = row.copy()
            row_ref["seg_start_refined"]  = row["seg_start"]
            row_ref["seg_length_refined"] = row["seg_length"]
            refined_rows.append(row_ref)
            continue

        new_start = overlap_1h["seg_start"].min()

        row_ref = row.copy()
        row_ref["seg_start_refined"]  = new_start
        row_ref["seg_length_refined"] = row_ref["seg_end"] - new_start
        refined_rows.append(row_ref)

    eps_big_refined = pd.DataFrame(refined_rows)
    combined = pd.concat([eps_big_refined, eps_1h], ignore_index=True)
    return combined

df_episodes_refined = refine_episodes_with_1h(
    episodes_all=auto_marked,
    episodes_1h=auto_1h_df,
)

print("Episodios refinados (incluye 1H):", df_episodes_refined.shape)
display(df_episodes_refined.head())


auto_marked cols: ['window', 'strategy', 'metric', 'variable', 'episode_id', 'seg_start', 'seg_end', 'seg_length', 'stat_max', 'n_windows', 'matched_manual', 'auto_len_sec']
auto_1h_df cols: ['window', 'strategy', 'metric', 'variable', 'episode_id', 'seg_start', 'seg_end', 'seg_length', 'stat_max', 'n_windows', 'matched_manual', 'auto_len_sec']
Episodios refinados (incluye 1H): (4860, 14)


,window,strategy,metric,variable,episode_id,seg_start,seg_end,seg_length,stat_max,n_windows,matched_manual,auto_len_sec,seg_start_refined,seg_length_refined
0,12H,decay,ks,var_1,1,2025-01-13 12:00:00,2025-01-17 12:00:00,4 days 00:00:00,0.760058,8,True,345600.0,2025-01-13 07:00:00,4 days 05:00:00
1,12H,decay,ks,var_1,2,2025-01-19 00:00:00,2025-01-20 12:00:00,1 days 12:00:00,0.721268,3,True,129600.0,2025-01-19 04:00:00,1 days 08:00:00
2,12H,decay,ks,var_10,1,2025-01-06 12:00:00,2025-01-09 00:00:00,2 days 12:00:00,0.848177,5,False,216000.0,2025-01-06 10:00:00,2 days 14:00:00
3,12H,decay,ks,var_10,2,2025-01-12 00:00:00,2025-01-20 12:00:00,8 days 12:00:00,0.788747,16,True,734400.0,2025-01-11 23:00:00,8 days 13:00:00
4,12H,decay,ks,var_2,1,2025-01-16 00:00:00,2025-01-19 12:00:00,3 days 12:00:00,0.827798,7,True,302400.0,2025-01-15 17:00:00,3 days 19:00:00


## 8. Re-evaluación con episodios refinados

In [21]:
df_eval_eps = df_episodes_refined.copy()

if "seg_start_refined" in df_eval_eps.columns:
    mask_has_ref = df_eval_eps["seg_start_refined"].notna()
    df_eval_eps.loc[mask_has_ref, "seg_start"] = df_eval_eps.loc[mask_has_ref, "seg_start_refined"]
    df_eval_eps["seg_length"] = df_eval_eps["seg_end"] - df_eval_eps["seg_start"]

cols_keep = [
    "window", "strategy", "metric", "variable",
    "seg_start", "seg_end", "episode_id"
]
df_eval_eps = df_eval_eps[cols_keep].copy()

print("df_eval_eps para evaluación refinada:", df_eval_eps.shape)
display(df_eval_eps.head())

def evaluate_episodes_vs_manual_multi_metric(
    df_episodes_auto: pd.DataFrame,
    intervals_manual: pd.DataFrame,
):
    if df_episodes_auto.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    man = intervals_manual.copy()
    total_days = max((t_max - t_min).total_seconds() / (3600 * 24), 1e-9)

    results = []
    marks_manual_all = []
    marks_auto_all   = []

    for (win, strat, metric), auto_sub in df_episodes_auto.groupby(
        ["window", "strategy", "metric"], dropna=False
    ):
        vars_in_auto = sorted(auto_sub["variable"].unique())
        man_sub = man[man["variable"].isin(vars_in_auto)].copy()
        if man_sub.empty and auto_sub.empty:
            continue

        manual_matches = []
        coverage_vals  = []
        delay_vals     = []

        for _, mrow in man_sub.iterrows():
            v  = mrow["variable"]
            ms = mrow["manual_start"]
            me = mrow["manual_end"]

            rel_auto = auto_sub[auto_sub["variable"] == v]

            total_overlap = pd.Timedelta(0)
            first_det = None

            for _, arow in rel_auto.iterrows():
                as_ = arow["seg_start"]
                ae  = arow["seg_end"]

                start = max(ms, as_)
                end   = min(me, ae)
                if end > start:
                    total_overlap += (end - start)

                    det_candidate = as_
                    if det_candidate < ms:
                        det_candidate = ms
                    if first_det is None or det_candidate < first_det:
                        first_det = det_candidate

            matched = total_overlap > pd.Timedelta(0)
            manual_matches.append(bool(matched))

            dur = me - ms
            if dur.total_seconds() > 0:
                cov_val = total_overlap.total_seconds() / dur.total_seconds()
            else:
                cov_val = 0.0
            coverage_vals.append(cov_val)

            if first_det is None:
                delay_vals.append(np.nan)
            else:
                first_det_shifted = first_det + pd.Timedelta(hours=6)
                delay = (first_det_shifted - ms).total_seconds() / 3600.0
                if delay < 0:
                    delay = 0.0
                delay_vals.append(delay)

        man_sub["matched_auto"] = manual_matches
        man_sub["coverage"]     = coverage_vals
        man_sub["delay_hours"]  = delay_vals

        TP = int(man_sub["matched_auto"].sum()) if not man_sub.empty else 0
        FN = int((~man_sub["matched_auto"]).sum()) if not man_sub.empty else 0

        auto_sub = auto_sub.copy()
        auto_matches = []
        for _, arow in auto_sub.iterrows():
            v  = arow["variable"]
            as_ = arow["seg_start"]
            ae  = arow["seg_end"]

            overlap = (
                (man_sub["variable"] == v) &
                ~(man_sub["manual_end"] < as_) &
                ~(man_sub["manual_start"] > ae)
            ).any()
            auto_matches.append(overlap)

        auto_sub["matched_manual"] = auto_matches
        FP = int((~auto_sub["matched_manual"]).sum()) if not auto_sub.empty else 0

        prec = TP / (TP + FP) if (TP + FP) > 0 else np.nan
        rec  = TP / (TP + FN) if (TP + FN) > 0 else np.nan

        if np.isnan(prec) or np.isnan(rec) or (prec + rec) == 0:
            f1 = np.nan
        else:
            f1 = 2 * prec * rec / (prec + rec)

        coverage_mean   = float(np.nanmean(coverage_vals))  if coverage_vals else np.nan
        coverage_median = float(np.nanmedian(coverage_vals)) if coverage_vals else np.nan

        delay_valid = [d for d in delay_vals if not np.isnan(d)]
        delay_mean_hours   = float(np.mean(delay_valid))   if delay_valid else np.nan
        delay_median_hours = float(np.median(delay_valid)) if delay_valid else np.nan

        false_alarms_per_day = FP / total_days

        if not man_sub.empty:
            man_sub["manual_len_sec"] = (
                man_sub["manual_end"] - man_sub["manual_start"]
            ).dt.total_seconds()
            manual_len_total_sec = man_sub["manual_len_sec"].sum()
        else:
            manual_len_total_sec = 0.0

        if not auto_sub.empty:
            auto_sub["auto_len_sec"] = (
                auto_sub["seg_end"] - auto_sub["seg_start"]
            ).dt.total_seconds()
            auto_len_total_sec = auto_sub["auto_len_sec"].sum()
        else:
            auto_len_total_sec = 0.0

        overlap_total_sec = 0.0
        if manual_len_total_sec > 0 and auto_len_total_sec > 0:
            for _, mrow in man_sub.iterrows():
                v  = mrow["variable"]
                ms = mrow["manual_start"]
                me = mrow["manual_end"]

                rel_auto = auto_sub[auto_sub["variable"] == v]
                for _, arow in rel_auto.iterrows():
                    as_ = arow["seg_start"]
                    ae  = arow["seg_end"]
                    start = max(ms, as_)
                    end   = min(me, ae)
                    if end > start:
                        overlap_total_sec += (end - start).total_seconds()

        if auto_len_total_sec > 0:
            prec_time = overlap_total_sec / auto_len_total_sec
        else:
            prec_time = np.nan

        if manual_len_total_sec > 0:
            rec_time = overlap_total_sec / manual_len_total_sec
        else:
            rec_time = np.nan

        if np.isnan(prec_time) or np.isnan(rec_time) or (prec_time + rec_time) == 0:
            f1_time = np.nan
        else:
            f1_time = 2 * prec_time * rec_time / (prec_time + rec_time)

        extra_time_sec = max(auto_len_total_sec - overlap_total_sec, 0.0)
        extra_hours = extra_time_sec / 3600.0 if extra_time_sec > 0 else 0.0

        if auto_len_total_sec > 0:
            extra_ratio_auto = extra_time_sec / auto_len_total_sec
        else:
            extra_ratio_auto = np.nan

        results.append({
            "window": win,
            "strategy": strat,
            "metric": metric,
            "TP_episodes": TP,
            "FP_episodes": FP,
            "FN_episodes": FN,
            "Precision": prec,
            "Recall": rec,
            "F1": f1,
            "manual_total_hours": manual_len_total_sec / 3600.0 if manual_len_total_sec > 0 else 0.0,
            "auto_total_hours": auto_len_total_sec / 3600.0 if auto_len_total_sec > 0 else 0.0,
            "overlap_hours": overlap_total_sec / 3600.0 if overlap_total_sec > 0 else 0.0,
            "Precision_time": prec_time,
            "Recall_time": rec_time,
            "F1_time": f1_time,
            "coverage_mean": coverage_mean,
            "coverage_median": coverage_median,
            "delay_mean_hours": delay_mean_hours,
            "delay_median_hours": delay_median_hours,
            "false_alarms_per_day": false_alarms_per_day,
            "extra_hours": extra_hours,
            "extra_ratio_auto": extra_ratio_auto,
        })

        man_sub["window"]   = win
        man_sub["strategy"] = strat
        man_sub["metric"]   = metric

        auto_sub["window"]   = win
        auto_sub["strategy"] = strat
        auto_sub["metric"]   = metric

        marks_manual_all.append(man_sub)
        marks_auto_all.append(auto_sub)

    eval_df_out = pd.DataFrame(results)
    manual_marked_out = (
        pd.concat(marks_manual_all, ignore_index=True)
        if marks_manual_all else pd.DataFrame()
    )
    auto_marked_out = (
        pd.concat(marks_auto_all, ignore_index=True)
        if marks_auto_all else pd.DataFrame()
    )

    return eval_df_out, manual_marked_out, auto_marked_out

eval_refined_df, manual_refined_df, auto_refined_df = evaluate_episodes_vs_manual_multi_metric(
    df_episodes_auto=df_eval_eps,
    intervals_manual=intervals_manual,
)

print("eval_refined_df shape:", eval_refined_df.shape)
display(eval_refined_df.head())

eval_refined_df = eval_refined_df.copy()
eval_refined_df["F1_time_filled"] = eval_refined_df["F1_time"].fillna(0.0)
eval_refined_df["Score_F1_combined"] = (
    0.5 * eval_refined_df["F1"] + 0.5 * eval_refined_df["F1_time_filled"]
)
display(eval_refined_df.head())


df_eval_eps para evaluación refinada: (4860, 7)


,window,strategy,metric,variable,seg_start,seg_end,episode_id
0,12H,decay,ks,var_1,2025-01-13 07:00:00,2025-01-17 12:00:00,1
1,12H,decay,ks,var_1,2025-01-19 04:00:00,2025-01-20 12:00:00,2
2,12H,decay,ks,var_10,2025-01-06 10:00:00,2025-01-09 00:00:00,1
3,12H,decay,ks,var_10,2025-01-11 23:00:00,2025-01-20 12:00:00,2
4,12H,decay,ks,var_2,2025-01-15 17:00:00,2025-01-19 12:00:00,1


eval_refined_df shape: (54, 22)


,window,strategy,metric,TP_episodes,FP_episodes,FN_episodes,Precision,Recall,F1,manual_total_hours,...,Precision_time,Recall_time,F1_time,coverage_mean,coverage_median,delay_mean_hours,delay_median_hours,false_alarms_per_day,extra_hours,extra_ratio_auto
0,12H,decay,ks,23,0,0,1.000000,1.0,1.000000,790.366667,...,0.465554,0.970731,0.629301,0.986019,1.0,6.478986,6.0,0.000000,880.766667,0.534446
1,12H,decay,psi,23,0,0,1.000000,1.0,1.000000,790.366667,...,0.459458,1.051031,0.639402,1.025774,1.0,6.136957,6.0,0.000000,977.300000,0.540542
2,12H,decay,wasserstein,23,0,0,1.000000,1.0,1.000000,790.366667,...,0.496267,0.947493,0.651368,0.934281,1.0,7.277536,6.0,0.000000,760.133333,0.503733
3,12H,golden,ks,23,1,0,0.958333,1.0,0.978723,790.366667,...,0.560559,0.892223,0.688532,0.932097,1.0,7.847826,6.0,0.050002,552.816667,0.439441
4,12H,golden,psi,23,1,0,0.958333,1.0,0.978723,790.366667,...,0.533361,0.798996,0.639699,0.885402,1.0,9.486232,6.0,0.050002,552.500000,0.466639


,window,strategy,metric,TP_episodes,FP_episodes,FN_episodes,Precision,Recall,F1,manual_total_hours,...,F1_time,coverage_mean,coverage_median,delay_mean_hours,delay_median_hours,false_alarms_per_day,extra_hours,extra_ratio_auto,F1_time_filled,Score_F1_combined
0,12H,decay,ks,23,0,0,1.000000,1.0,1.000000,790.366667,...,0.629301,0.986019,1.0,6.478986,6.0,0.000000,880.766667,0.534446,0.629301,0.814651
1,12H,decay,psi,23,0,0,1.000000,1.0,1.000000,790.366667,...,0.639402,1.025774,1.0,6.136957,6.0,0.000000,977.300000,0.540542,0.639402,0.819701
2,12H,decay,wasserstein,23,0,0,1.000000,1.0,1.000000,790.366667,...,0.651368,0.934281,1.0,7.277536,6.0,0.000000,760.133333,0.503733,0.651368,0.825684
3,12H,golden,ks,23,1,0,0.958333,1.0,0.978723,790.366667,...,0.688532,0.932097,1.0,7.847826,6.0,0.050002,552.816667,0.439441,0.688532,0.833628
4,12H,golden,psi,23,1,0,0.958333,1.0,0.978723,790.366667,...,0.639699,0.885402,1.0,9.486232,6.0,0.050002,552.500000,0.466639,0.639699,0.809211


## 9. Resúmenes agregados (caso REFINADO con 1H)

In [22]:
cols_summary_ref = [
    c for c in [
        "F1", "F1_time", "delay_mean_hours",
        "extra_ratio_auto", "false_alarms_per_day",
        "Score_F1_combined"
    ]
    if c in eval_refined_df.columns
]

summary_full_ref = (
    eval_refined_df
    .groupby(["metric", "window", "strategy"])[cols_summary_ref]
    .mean()
    .reset_index()
)

summary_full_ref["metric"]   = pd.Categorical(summary_full_ref["metric"],   categories=method_order,   ordered=True)
summary_full_ref["window"]   = pd.Categorical(summary_full_ref["window"],   categories=window_order,   ordered=True)
summary_full_ref["strategy"] = pd.Categorical(summary_full_ref["strategy"], categories=strategy_order, ordered=True)

summary_full_ref = summary_full_ref.sort_values(["metric", "window", "strategy"])

print("📘 Resumen REFINADO por método, ventana y estrategia")
display(summary_full_ref.head(40))

summary_by_window_ref = (
    summary_full_ref
    .groupby("window")[cols_summary_ref]
    .mean()
    .reset_index()
    .sort_values("window", key=lambda s: s.astype(str).str.rstrip("H").astype(int))
)

print("\n📌 Promedios REFINADOS por ventana:")
display(summary_by_window_ref)

📘 Resumen REFINADO por método, ventana y estrategia


,metric,window,strategy,F1,F1_time,delay_mean_hours,extra_ratio_auto,false_alarms_per_day,Score_F1_combined
33,psi,6H,decay,0.958333,0.632896,6.922464,0.544370,0.100003,0.795615
35,psi,6H,seasonal,0.958333,0.633460,6.229710,0.547588,0.100003,0.795897
34,psi,6H,golden,0.851852,0.688643,8.889855,0.468512,0.400014,0.770247
18,psi,12H,decay,1.000000,0.639402,6.136957,0.540542,0.000000,0.819701
20,psi,12H,seasonal,1.000000,0.624042,6.160870,0.542228,0.000000,0.812021
19,psi,12H,golden,0.978723,0.639699,9.486232,0.466639,0.050002,0.809211
24,psi,24H,decay,0.977778,0.595002,7.415909,0.558067,0.000000,0.786390
26,psi,24H,seasonal,0.977778,0.591091,7.415909,0.562368,0.000000,0.784435
25,psi,24H,golden,0.977778,0.642566,6.970455,0.490200,0.000000,0.810172
27,psi,36H,decay,1.000000,0.557421,6.229710,0.610203,0.000000,0.778711



📌 Promedios REFINADOS por ventana:


C:\Users\frncc\AppData\Local\Temp\ipykernel_15460\3011481604.py:28: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,window,F1,F1_time,delay_mean_hours,extra_ratio_auto,false_alarms_per_day,Score_F1_combined
0,6H,0.889774,0.640210,7.421337,0.511749,0.322233,0.764992
1,12H,0.992908,0.646524,7.306361,0.501499,0.016667,0.819716
2,24H,0.975416,0.624350,7.095118,0.522369,0.005556,0.799883
3,36H,1.000000,0.589982,6.550242,0.573257,0.000000,0.794991
4,48H,0.904762,0.542014,6.230799,0.588633,0.000000,0.723388


## 10. Comparación directa BASE vs REFINADO

In [23]:
compare_cols = ["F1", "F1_time", "delay_mean_hours", "extra_ratio_auto", "Score_F1_combined"]

base_key = summary_full_base[["metric", "window", "strategy"] + compare_cols].copy()
ref_key  = summary_full_ref[["metric", "window", "strategy"] + compare_cols].copy()

base_key = base_key.add_suffix("_base")
ref_key  = ref_key.add_suffix("_ref")

merge_df = base_key.merge(
    ref_key,
    left_on=["metric_base", "window_base", "strategy_base"],
    right_on=["metric_ref", "window_ref", "strategy_ref"],
    how="inner"
)

for col in compare_cols:
    merge_df[f"delta_{col}"] = merge_df[f"{col}_ref"] - merge_df[f"{col}_base"]

print("📊 Comparación BASE vs REFINADO (primeras filas)")
display(merge_df.head(40))

📊 Comparación BASE vs REFINADO (primeras filas)


,metric_base,window_base,strategy_base,F1_base,F1_time_base,delay_mean_hours_base,extra_ratio_auto_base,Score_F1_combined_base,metric_ref,window_ref,...,F1_ref,F1_time_ref,delay_mean_hours_ref,extra_ratio_auto_ref,Score_F1_combined_ref,delta_F1,delta_F1_time,delta_delay_mean_hours,delta_extra_ratio_auto,delta_Score_F1_combined
0,psi,6H,decay,0.893617,0.631283,7.903968,0.525053,0.762450,psi,6H,...,0.958333,0.632896,6.922464,0.544370,0.795615,6.471631e-02,0.001613,-0.981504,0.019317,0.033165
1,psi,6H,seasonal,0.893617,0.627913,7.285714,0.532870,0.760765,psi,6H,...,0.958333,0.633460,6.229710,0.547588,0.795897,6.471631e-02,0.005547,-1.056004,0.014718,0.035132
2,psi,6H,golden,0.750000,0.627031,10.761111,0.469123,0.688516,psi,6H,...,0.851852,0.688643,8.889855,0.468512,0.770247,1.018519e-01,0.061612,-1.871256,-0.000611,0.081732
3,psi,12H,decay,0.956522,0.631121,7.655303,0.524562,0.793822,psi,12H,...,1.000000,0.639402,6.136957,0.540542,0.819701,4.347826e-02,0.008280,-1.518347,0.015980,0.025879
4,psi,12H,seasonal,0.956522,0.624742,7.655303,0.531766,0.790632,psi,12H,...,1.000000,0.624042,6.160870,0.542228,0.812021,4.347826e-02,-0.000700,-1.494433,0.010463,0.021389
5,psi,12H,golden,0.936170,0.642938,11.362879,0.437926,0.789554,psi,12H,...,0.978723,0.639699,9.486232,0.466639,0.809211,4.255319e-02,-0.003240,-1.876647,0.028712,0.019657
6,psi,24H,decay,0.977778,0.580732,9.097727,0.562521,0.779255,psi,24H,...,0.977778,0.595002,7.415909,0.558067,0.786390,1.110223e-16,0.014270,-1.681818,-0.004454,0.007135
7,psi,24H,seasonal,0.977778,0.574862,9.097727,0.569150,0.776320,psi,24H,...,0.977778,0.591091,7.415909,0.562368,0.784435,1.110223e-16,0.016229,-1.681818,-0.006782,0.008115
8,psi,24H,golden,0.954545,0.643511,8.830159,0.470479,0.799028,psi,24H,...,0.977778,0.642566,6.970455,0.490200,0.810172,2.323232e-02,-0.000944,-1.859704,0.019721,0.011144
9,psi,36H,decay,0.977778,0.555165,8.184848,0.602923,0.766471,psi,36H,...,1.000000,0.557421,6.229710,0.610203,0.778711,2.222222e-02,0.002256,-1.955138,0.007280,0.012239


In [24]:
# %% 11. Visualizaciones compactas (caso REFINADO con 1H)

# mismos órdenes y labels que antes
window_order   = sorted(eval_refined_df["window"].astype(str).unique(),
                        key=lambda w: int(str(w).rstrip("H")))
method_order   = ["psi", "ks", "wasserstein"]
strategy_order = ["decay", "seasonal", "golden"]

method_labels = {
    "psi": "PSI",
    "ks": "Kolmogorov–Smirnov",
    "wasserstein": "Wasserstein",
}

eval_plot_ref = eval_refined_df.copy()
eval_plot_ref = eval_plot_ref[eval_plot_ref["metric"].isin(method_order)].copy()
eval_plot_ref["method_label"] = eval_plot_ref["metric"].map(method_labels)

eval_plot_ref["window"] = pd.Categorical(
    eval_plot_ref["window"].astype(str),
    categories=window_order,
    ordered=True
)
eval_plot_ref["strategy"] = pd.Categorical(
    eval_plot_ref["strategy"],
    categories=strategy_order,
    ordered=True
)
eval_plot_ref["method_label"] = pd.Categorical(
    eval_plot_ref["method_label"],
    categories=[method_labels[m] for m in method_order],
    ordered=True
)

# --- 11.1 Score combinado refinado ---
fig_score_ref = px.bar(
    eval_plot_ref,
    x="window",
    y="Score_F1_combined",
    color="strategy",
    barmode="group",
    facet_col="method_label",
    category_orders={
        "window": window_order,
        "strategy": strategy_order,
        "method_label": [method_labels[m] for m in method_order],
    },
    labels={
        "window": "Ventana",
        "Score_F1_combined": "Score combinado (F1 / F1_time)",
        "strategy": "Estrategia",
        "method_label": "Método estadístico",
    },
    title="Score combinado (REFINADO con 1H) por ventana, estrategia y método",
)
fig_score_ref.update_yaxes(matches=None)
fig_score_ref.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_score_ref.show()

# --- 11.2 Delay medio refinado ---
fig_delay_ref = px.bar(
    eval_plot_ref,
    x="window",
    y="delay_mean_hours",
    color="strategy",
    barmode="group",
    facet_col="method_label",
    category_orders={
        "window": window_order,
        "strategy": strategy_order,
        "method_label": [method_labels[m] for m in method_order],
    },
    labels={
        "window": "Ventana",
        "delay_mean_hours": "Delay medio (horas)",
        "strategy": "Estrategia",
        "method_label": "Método estadístico",
    },
    title="Delay medio (REFINADO con 1H) por ventana, estrategia y método",
)
fig_delay_ref.update_yaxes(matches=None)
fig_delay_ref.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_delay_ref.show()

# --- 11.3 Proporción de tiempo extra refinado ---
fig_extra_ref = px.bar(
    eval_plot_ref,
    x="window",
    y="extra_ratio_auto",
    color="strategy",
    barmode="group",
    facet_col="method_label",
    category_orders={
        "window": window_order,
        "strategy": strategy_order,
        "method_label": [method_labels[m] for m in method_order],
    },
    labels={
        "window": "Ventana",
        "extra_ratio_auto": "Proporción de tiempo extra",
        "strategy": "Estrategia",
        "method_label": "Método estadístico",
    },
    title="Tiempo extra (REFINADO con 1H) por ventana, estrategia y método",
)
fig_extra_ref.update_yaxes(matches=None)
fig_extra_ref.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_extra_ref.show()


In [25]:
# %% 12. Plots de mejora (BASE vs REFINADO)

comp = merge_df.copy()

# columnas "limpias" para plotear
comp["metric"]   = comp["metric_base"]
comp["window"]   = comp["window_base"].astype(str)
comp["strategy"] = comp["strategy_base"]

# orden de categorías
comp["window"] = pd.Categorical(comp["window"],
                                categories=window_order,
                                ordered=True)
comp["strategy"] = pd.Categorical(comp["strategy"],
                                  categories=strategy_order,
                                  ordered=True)
comp["metric"] = pd.Categorical(comp["metric"],
                                categories=method_order,
                                ordered=True)

comp["method_label"] = comp["metric"].map(method_labels)

# --- 12.1 Mejora en Score combinado ---
fig_delta_score = px.bar(
    comp,
    x="window",
    y="delta_Score_F1_combined",
    color="strategy",
    barmode="group",
    facet_col="method_label",
    category_orders={
        "window": window_order,
        "strategy": strategy_order,
        "method_label": [method_labels[m] for m in method_order],
    },
    labels={
        "window": "Ventana",
        "delta_Score_F1_combined": "Δ Score combinado (refinado - base)",
        "strategy": "Estrategia",
        "method_label": "Método estadístico",
    },
    title="Mejora en Score combinado al usar 1H",
)
fig_delta_score.update_yaxes(matches=None)
fig_delta_score.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_delta_score.show()

# --- 12.2 Mejora en F1 al usar 1H ---
fig_delta_f1time = px.bar(
    comp,
    x="window",
    y="delta_F1",
    color="strategy",
    barmode="group",
    facet_col="method_label",
    category_orders={
        "window": window_order,
        "strategy": strategy_order,
        "method_label": [method_labels[m] for m in method_order],
    },
    labels={
        "window": "Ventana",
        "delta_F1_time": "Δ F1_time (refinado - base)",
        "strategy": "Estrategia",
        "method_label": "Método estadístico",
    },
    title="Mejora en F1 al usar 1H",
)
fig_delta_f1time.update_yaxes(matches=None)
fig_delta_f1time.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_delta_f1time.show()
# --- 12.3 Mejora en F1_time al usar 1H ---
fig_delta_f1time = px.bar(
    comp,
    x="window",
    y="delta_F1_time",
    color="strategy",
    barmode="group",
    facet_col="method_label",
    category_orders={
        "window": window_order,
        "strategy": strategy_order,
        "method_label": [method_labels[m] for m in method_order],
    },
    labels={
        "window": "Ventana",
        "delta_F1_time": "Δ F1_time (refinado - base)",
        "strategy": "Estrategia",
        "method_label": "Método estadístico",
    },
    title="Mejora en F1_time al usar 1H",
)
fig_delta_f1time.update_yaxes(matches=None)
fig_delta_f1time.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_delta_f1time.show()

# --- 12.3 Cambio en delay medio ---
fig_delta_delay = px.bar(
    comp,
    x="window",
    y="delta_delay_mean_hours",
    color="strategy",
    barmode="group",
    facet_col="method_label",
    category_orders={
        "window": window_order,
        "strategy": strategy_order,
        "method_label": [method_labels[m] for m in method_order],
    },
    labels={
        "window": "Ventana",
        "delta_delay_mean_hours": "Δ Delay medio (horas)",
        "strategy": "Estrategia",
        "method_label": "Método estadístico",
    },
    title="Cambio en delay medio al usar 1H (valores negativos = más rápido)",
)
fig_delta_delay.update_yaxes(matches=None)
fig_delta_delay.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_delta_delay.show()

# --- 12.4 Cambio en proporción de tiempo extra ---
fig_delta_extra = px.bar(
    comp,
    x="window",
    y="delta_extra_ratio_auto",
    color="strategy",
    barmode="group",
    facet_col="method_label",
    category_orders={
        "window": window_order,
        "strategy": strategy_order,
        "method_label": [method_labels[m] for m in method_order],
    },
    labels={
        "window": "Ventana",
        "delta_extra_ratio_auto": "Δ Proporción tiempo extra",
        "strategy": "Estrategia",
        "method_label": "Método estadístico",
    },
    title="Cambio en tiempo extra al usar 1H (valores negativos = menos falsos positivos)",
)
fig_delta_extra.update_yaxes(matches=None)
fig_delta_extra.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig_delta_extra.show()


# Visualización

In [26]:
import matplotlib.pyplot as plt

def plot_grid_episodes_matplotlib(
    df,
    df_episodes_auto: pd.DataFrame,
    intervals_manual: pd.DataFrame,
    strategy: str,
    window: str,
    metric: str = "psi",
    show_manual: bool = True,
    title_prefix: str = "Episodios detectados",
):
    vars10 = list(df.columns)[:10]
    rows, cols = 5, 2

    subset_auto = df_episodes_auto[
        (df_episodes_auto["strategy"] == strategy) &
        (df_episodes_auto["window"] == window) &
        (df_episodes_auto["metric"] == metric)
    ]

    fig, axes = plt.subplots(rows, cols, figsize=(12, 16), sharex=True)
    axes = axes.flatten()

    for i, var in enumerate(vars10):
        ax = axes[i]
        s = df[var]

        ax.plot(s.index, s.values, linewidth=0.8)
        ax.set_title(var, fontsize=9)

        if show_manual:
            mans = intervals_manual[intervals_manual["variable"] == var]
            for _, m in mans.iterrows():
                ax.axvspan(
                    m["manual_start"], m["manual_end"],
                    alpha=0.18, color="red"
                )

        autos = subset_auto[subset_auto["variable"] == var]
        for _, a in autos.iterrows():
            ax.axvspan(
                a["seg_start"], a["seg_end"],
                alpha=0.18, color="blue"
            )

        ax.grid(True, alpha=0.3)

    # ejes vacíos si hay menos de 10 variables
    for j in range(len(vars10), len(axes)):
        fig.delaxes(axes[j])

    fig.suptitle(
        f"{title_prefix} — strategy={strategy}, window={window}, metric={metric}\n"
        f"(rojo = manual, azul = auto)",
        fontsize=12
    )
    fig.tight_layout(rect=[0, 0, 1, 0.95])

    return fig

# %%  🚀 Exportar episodios REFINADOS
# Usamos df_eval_eps, que ya tiene seg_start ajustado (si existía seg_start_refined)
df_episodes_auto_refined = df_eval_eps.copy()
df_episodes_auto_refined["window"] = df_episodes_auto_refined["window"].astype(str)

# Carpeta para plots refinados dentro de synthetic_data/results
EXPORT_ROOT_REFINED = RESULTS_1H_DIR / "plots_refined"
EXPORT_ROOT_REFINED.mkdir(parents=True, exist_ok=True)

PLOT_WINDOWS_REFINED   = sorted(
    df_episodes_auto_refined["window"].unique(),
    key=lambda w: int(str(w).rstrip("H"))
)
PLOT_STRATEGIES_REFINED = sorted(df_episodes_auto_refined["strategy"].unique())
PLOT_METRICS_REFINED    = sorted(df_episodes_auto_refined["metric"].unique())

print("Ventanas refinadas:", PLOT_WINDOWS_REFINED)
print("Estrategias refinadas:", PLOT_STRATEGIES_REFINED)
print("Métricas refinadas:", PLOT_METRICS_REFINED)

print("\nExportando combinaciones REFINADAS (ventanas grandes + 1H) a PNG")

for win in PLOT_WINDOWS_REFINED:
    win_dir = EXPORT_ROOT_REFINED / f"window_{win}"
    win_dir.mkdir(parents=True, exist_ok=True)

    for strat in PLOT_STRATEGIES_REFINED:
        for metric in PLOT_METRICS_REFINED:
            subset_auto = df_episodes_auto_refined[
                (df_episodes_auto_refined["strategy"] == strat) &
                (df_episodes_auto_refined["window"]   == win) &
                (df_episodes_auto_refined["metric"]   == metric)
            ]
            if subset_auto.empty:
                continue

            print("\n==============================================")
            print(f"[REFINADO] Ventana: {win} | Estrategia: {strat} | Métrica: {metric}")
            print("==============================================")

            fig = plot_grid_episodes_matplotlib(
                df=df,
                df_episodes_auto=subset_auto,
                intervals_manual=intervals_manual,
                strategy=strat,
                window=win,
                metric=metric,
                show_manual=True,
                title_prefix="Episodios REFINADOS (ventanas + 1H)",
            )

            out_path = win_dir / f"episodes_{metric}_{strat}_{win}_refinado.png"
            fig.savefig(out_path, dpi=200, bbox_inches="tight")
            plt.close(fig)

            print(f"[OK] Exportado: {out_path}")

Ventanas refinadas: ['1H', '6H', '12H', '24H', '36H', '48H']
Estrategias refinadas: ['decay', 'golden', 'seasonal']
Métricas refinadas: ['ks', 'psi', 'wasserstein']

Exportando combinaciones REFINADAS (ventanas grandes + 1H) a PNG

[REFINADO] Ventana: 1H | Estrategia: decay | Métrica: ks
[OK] Exportado: synthetic_data\results_1H\plots_refined\window_1H\episodes_ks_decay_1H_refinado.png

[REFINADO] Ventana: 1H | Estrategia: decay | Métrica: psi
[OK] Exportado: synthetic_data\results_1H\plots_refined\window_1H\episodes_psi_decay_1H_refinado.png

[REFINADO] Ventana: 1H | Estrategia: decay | Métrica: wasserstein
[OK] Exportado: synthetic_data\results_1H\plots_refined\window_1H\episodes_wasserstein_decay_1H_refinado.png

[REFINADO] Ventana: 1H | Estrategia: golden | Métrica: ks
[OK] Exportado: synthetic_data\results_1H\plots_refined\window_1H\episodes_ks_golden_1H_refinado.png

[REFINADO] Ventana: 1H | Estrategia: golden | Métrica: psi
[OK] Exportado: synthetic_data\results_1H\plots_refined